In [ ]:
pip install ultralytics==8.3.32 pyyaml tqdm

In [ ]:
# train_pig_yolov8x.py
from ultralytics import YOLO
from pathlib import Path
import yaml

DATA_ROOT = Path("/path/to/your/rf_yolov8_dataset")
DATA_YAML = next(DATA_ROOT.rglob("data.yaml"))
MODEL     = "yolov8x.pt"
SEED      = 42

# Phase A
model = YOLO(MODEL)
train_res = model.train(
    data=str(DATA_YAML),
    project="runs_pig_detect_local", name="phaseA",
    imgsz=1536, epochs=150, patience=30,
    device=0, workers=4, seed=SEED,
    cos_lr=True, close_mosaic=20,

    # augment
    degrees=8, translate=0.10, scale=0.20, shear=3,
    fliplr=0.5, flipud=0.0,
    hsv_h=0.015, hsv_s=0.35, hsv_v=0.20,
    mosaic=0.4, mixup=0.10, copy_paste=0.0,
    batch=-1,  # auto batch size theo VRAM
)
best_a = Path(train_res.save_dir) / "weights" / "best.pt"

# Phase B (fine-tune)
model_ft = YOLO(str(best_a))
ft_res = model_ft.train(
    data=str(DATA_YAML),
    project="runs_pig_detect_local", name="phaseB_finetune",
    imgsz=1792, epochs=40, patience=20,
    device=0, workers=4, seed=SEED,
    lr0=0.001, lrf=0.01, cos_lr=True,
    close_mosaic=10,

    degrees=5, translate=0.07, scale=0.15, shear=2,
    fliplr=0.5, mosaic=0.1, mixup=0.05, copy_paste=0.0,
    batch=-1,
)

# Final val
final_pt = Path(ft_res.save_dir) / "weights" / "best.pt"
final_model = YOLO(str(final_pt))
final_val = final_model.val(data=str(DATA_YAML), imgsz=1792, device=0)
print("Final metrics:", final_val.results_dict)
print("Best weights:", final_pt)
